In [1]:
# Block 1: Imports and Load Data
import pandas as pd
import numpy as np

print("Loading Dimensions Data...\n")

# Load the raw data
df_emp = pd.read_csv("../HR data/Raw/Dim_Employees.csv")
df_roles = pd.read_csv("../HR data/Raw/Dim_Job_Roles.csv")

print(f"Dim_Employees loaded with {df_emp.shape[0]:,} rows.")
print(f"Dim_Job_Roles loaded with {df_roles.shape[0]} rows.")

Loading Dimensions Data...

Dim_Employees loaded with 15,000 rows.
Dim_Job_Roles loaded with 77 rows.


In [2]:
df_emp.head()

,employee_id,first_name,last_name,gender,birth_date,hire_date,exit_date,attrition_flag,manager_id,role_id
0,1,Karim,Ghoneim,Male,07/29/1998,10/17/2022,NaN,No,8785,57
1,2,rabab,Zaki,Female,06/14/1977,04/21/2020,NaN,No,10228,63
2,3,Samy,Shalaby,Male,08/10/1996,07/23/2020,06/10/2025,Yes,12654,47
3,4,Tarek,Nasr,Male,09/28/1976,03/16/2020,NaN,No,7039,74
4,5,Heba,Khamis,Female,10/08/1996,06/25/2020,01/08/2024,Yes,4626,69


In [3]:
# Basic Cleaning for Dim_Employees
print("Cleaning basic text and dates in Dim_Employees...")

# Fix column name (remove trailing spaces like 'employee_id ')
df_emp.rename(columns=lambda x: x.strip(), inplace=True)

# Standardize text (Title Case for names)
df_emp['first_name'] = df_emp['first_name'].str.title()
df_emp['last_name'] = df_emp['last_name'].str.title()

# Convert dates to datetime objects
date_cols = ['birth_date', 'hire_date', 'exit_date']
for col in date_cols:
    df_emp[col] = pd.to_datetime(df_emp[col], errors='coerce')

print("Column names, text cases, and dates are standardized.")

Cleaning basic text and dates in Dim_Employees...
Column names, text cases, and dates are standardized.


In [4]:
# Creating Data Quality Flags (Flag First, Delete Later)
print("Creating Data Quality Flags for Dim_Employees...")

# Flag 1: Ghost Managers (manager_id not in employee_id list)
valid_employee_ids = df_emp['employee_id'].unique()
df_emp['is_ghost_manager_flag'] = ~df_emp['manager_id'].isin(valid_employee_ids)

# Flag 2: Illogical Exit Dates (Exit before Hire)
df_emp['invalid_exit_date_flag'] = df_emp['exit_date'] < df_emp['hire_date']

# Flag 3: Missing Exit Date for Leavers
df_emp['missing_exit_date_flag'] = (df_emp['attrition_flag'] == 'Yes') & (df_emp['exit_date'].isna())

# Flag 4: Active employee but has an exit date
df_emp['active_with_exit_date_flag'] = (df_emp['attrition_flag'] == 'No') & (df_emp['exit_date'].notna())

print(f"   -> {df_emp['is_ghost_manager_flag'].sum()} ghost managers found.")
print(f"   -> {df_emp['invalid_exit_date_flag'].sum()} invalid exit dates (before hire).")
print(f"   -> {df_emp['missing_exit_date_flag'].sum()} leavers missing an exit date.")
print("Flags created successfully.")

Creating Data Quality Flags for Dim_Employees...
   -> 60 ghost managers found.
   -> 45 invalid exit dates (before hire).
   -> 152 leavers missing an exit date.
Flags created successfully.


In [5]:
df_emp.head()

,employee_id,first_name,last_name,gender,birth_date,hire_date,exit_date,attrition_flag,manager_id,role_id,is_ghost_manager_flag,invalid_exit_date_flag,missing_exit_date_flag,active_with_exit_date_flag
0,1,Karim,Ghoneim,Male,1998-07-29,2022-10-17,NaT,No,8785,57,False,False,False,False
1,2,Rabab,Zaki,Female,1977-06-14,2020-04-21,NaT,No,10228,63,False,False,False,False
2,3,Samy,Shalaby,Male,1996-08-10,2020-07-23,2025-06-10,Yes,12654,47,False,False,False,False
3,4,Tarek,Nasr,Male,1976-09-28,2020-03-16,NaT,No,7039,74,False,False,False,False
4,5,Heba,Khamis,Female,1996-10-08,2020-06-25,2024-01-08,Yes,4626,69,False,False,False,False


In [6]:
# Cleaning Dim_Job_Roles
print("Cleaning Dim_Job_Roles...")

# Extract Min and Max salary from base_salary_range
# Example: "8000-15000" -> min_salary: 8000, max_salary: 15000
df_roles[['min_salary', 'max_salary']] = df_roles['base_salary_range'].str.split('-', expand=True).astype(float)

print("Successfully extracted min_salary and max_salary.")
# Show the first 3 rows to verify
display(df_roles.head(3))

Cleaning Dim_Job_Roles...
Successfully extracted min_salary and max_salary.


,role_id,job_title,department,job_level,base_salary_range,min_salary,max_salary
0,1,Junior IT Support,IT,Junior,8000-15000,8000.0,15000.0
1,2,Jr. Data Analyst,IT,Junior,8000-15000,8000.0,15000.0
2,3,Systems Analyst,IT,Mid,15000-25000,15000.0,25000.0


In [7]:
# Save Cleaned Dimensions
print("Saving cleaned dimensions to CSV...")

# Save the cleaned datasets
df_emp.to_csv("Dim_Employees_Cleaned.csv", index=False)
df_roles.to_csv("Dim_Job_Roles_Cleaned.csv", index=False)

print("Dimensions saved successfully as 'Dim_Employees_Cleaned.csv' and 'Dim_Job_Roles_Cleaned.csv'.")

Saving cleaned dimensions to CSV...
Dimensions saved successfully as 'Dim_Employees_Cleaned.csv' and 'Dim_Job_Roles_Cleaned.csv'.
